# Module 28: Benchmarking PyTorch — Measuring Performance Correctly

**torch.utils.benchmark Deep Dive**

| | |
|---|---|
| **Prerequisites** | Module 07 (Training), Module 08 (torch.compile) |
| **Time** | ~2 hours |
| **Notebook** | 28 of 28 |

---

## What You'll Learn

1. Why `time.time()` and `timeit` are wrong for PyTorch
2. `torch.utils.benchmark.Timer` — the correct approach
3. `blocked_autorange()` — automatic iteration tuning
4. `Measurement` objects — accessing raw timing data
5. `Compare` tables — organized side-by-side comparison
6. Shape sweep benchmarks
7. Benchmarking `torch.compile` properly
8. `num_threads` control
9. `Fuzzer` for random configurations
10. Common pitfalls and how to avoid them

In [ ]:
import torch
import torch.nn as nn
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CPU threads: {torch.get_num_threads()}")

## 1. Why `time.time()` Is Wrong

Most benchmarks you'll find online use `time.time()` or `timeit`. Both are problematic for PyTorch:

- **CUDA is async**: GPU ops return immediately; `time.time()` measures launch time, not compute
- **No warmup**: First call initializes contexts, JIT compiles, triggers autotuning
- **GC interference**: Garbage collection pauses inject random latency
- **No statistical rigor**: Single measurements are noisy

`torch.utils.benchmark` handles all of this automatically.

In [ ]:
# Demonstration: time.time() is noisy and unreliable
x = torch.randn(1000, 1000)
y = torch.randn(1000, 1000)

naive_times = []
for _ in range(20):
    start = time.time()
    _ = x @ y
    naive_times.append(time.time() - start)

print("time.time() results (20 runs):")
print(f"  Mean:   {sum(naive_times)/len(naive_times)*1e3:.3f} ms")
print(f"  Min:    {min(naive_times)*1e3:.3f} ms")
print(f"  Max:    {max(naive_times)*1e3:.3f} ms")
print(f"  Spread: {(max(naive_times)-min(naive_times))*1e3:.3f} ms")
print(f"  Spread is {(max(naive_times)-min(naive_times))/min(naive_times)*100:.0f}% of min")

## 2. Timer Basics

The core API: `torch.utils.benchmark.Timer`

Key parameters:
- `stmt`: Code to benchmark
- `setup`: Code run once before measurement
- `globals`: Python objects accessible in stmt/setup
- `num_threads`: Pin CPU thread count
- `label`, `sub_label`, `description`: For Compare tables

In [ ]:
from torch.utils.benchmark import Timer

# Option 1: setup string
t1 = Timer(
    stmt="x @ y",
    setup="import torch; x = torch.randn(256, 256); y = torch.randn(256, 256)",
)
print("With setup string:")
print(t1.timeit(100))

# Option 2: globals dict (more flexible)
x = torch.randn(256, 256)
y = torch.randn(256, 256)
t2 = Timer(
    stmt="x @ y",
    globals={"x": x, "y": y},
)
print("\nWith globals dict:")
print(t2.timeit(100))

## 3. `blocked_autorange()` — The Recommended Approach

Instead of choosing a fixed iteration count, `blocked_autorange()` automatically:
1. Runs increasing iterations (1, 2, 4, 8, ...) until wall time exceeds `min_run_time`
2. Runs multiple blocks at the determined count
3. Reports **median** (not mean) — robust to outliers

In [ ]:
t = Timer(
    stmt="x @ y",
    globals={"x": x, "y": y},
)

# timeit: fixed count, you guess the right number
fixed = t.timeit(50)
print("timeit(50):")
print(fixed)

# blocked_autorange: auto-determined, statistically robust
auto = t.blocked_autorange(min_run_time=1.0)
print("\nblocked_autorange(min_run_time=1.0):")
print(auto)
print(f"  Auto-selected {auto.number_per_run} runs per block")
print(f"  Collected {len(auto.raw_times)} block(s)")

## 4. Measurement Object

Both `timeit()` and `blocked_autorange()` return a `Measurement` with rich timing data.

In [ ]:
t = Timer(
    stmt="torch.nn.functional.relu(x)",
    globals={"x": torch.randn(1000, 1000), "torch": torch},
)
m = t.blocked_autorange(min_run_time=1.0)

print(f"Mean:            {m.mean * 1e6:.2f} us")
print(f"Median:          {m.median * 1e6:.2f} us")
print(f"IQR:             {m.iqr * 1e6:.2f} us")
print(f"Number per run:  {m.number_per_run}")
print(f"Raw blocks:      {len(m.raw_times)}")
print(f"Per-run samples: {len(m.times)}")

print("\nPer-run times (us):")
for i, t_val in enumerate(m.times[:10]):
    print(f"  [{i:2d}] {t_val * 1e6:.2f}")

## 5. Compare Tables — Side-by-Side

The `Compare` class renders formatted tables comparing multiple benchmarks.

Label hierarchy:
- `label` → section header
- `sub_label` → row within section
- `description` → column

In [ ]:
from torch.utils.benchmark import Compare

results = []
for n in [64, 256, 1024]:
    a = torch.randn(n, n)
    b = torch.randn(n, n)

    for desc, stmt in [
        ("mm", "a.mm(b)"),
        ("matmul", "a @ b"),
        ("einsum", "torch.einsum('ij,jk->ik', a, b)"),
    ]:
        t = Timer(
            stmt=stmt,
            globals={"a": a, "b": b, "torch": torch},
            label="Matrix multiply",
            sub_label=f"[{n}x{n}]",
            description=desc,
            num_threads=1,
        )
        results.append(t.blocked_autorange(min_run_time=0.5))

compare = Compare(results)
compare.print()

In [ ]:
# Colorized and trimmed
compare = Compare(results)
compare.trim_significant_figures()
compare.colorize()
compare.print()

## 6. Shape Sweep Benchmark

Sweep over input sizes to understand scaling behavior.

In [ ]:
sizes = [128, 256, 512, 1024, 2048]
sweep_results = []

for n in sizes:
    a = torch.randn(n, n)
    b = torch.randn(n, n)
    t = Timer(
        stmt="a @ b",
        globals={"a": a, "b": b},
        label="matmul scaling",
        sub_label=f"[{n}x{n}]",
        description="mm",
        num_threads=1,
    )
    sweep_results.append(t.blocked_autorange(min_run_time=0.5))

Compare(sweep_results).print()

print("\nScaling analysis (matmul is O(n^3)):")
for i in range(1, len(sizes)):
    time_ratio = sweep_results[i].median / sweep_results[i - 1].median
    theoretical = (sizes[i] / sizes[i - 1]) ** 3
    print(f"  {sizes[i]:4d} vs {sizes[i-1]:4d}: "
          f"actual={time_ratio:.1f}x, theoretical O(n^3)={theoretical:.1f}x")

## 7. Benchmarking `torch.compile` Properly

**Critical**: The first call to a compiled function triggers compilation (seconds).
You MUST warm up the compiled model OUTSIDE the Timer.

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, x):
        return self.net(x)

dim = 512
model = SimpleModel(dim)
x = torch.randn(64, dim)

# Eager baseline
eager_timer = Timer(
    stmt="model(x)",
    globals={"model": model, "x": x},
    label="FFN",
    sub_label=f"dim={dim}",
    description="eager",
    num_threads=1,
)

# Compiled — warmup OUTSIDE the Timer
compiled = torch.compile(model)
print("Warming up compiled model...")
for _ in range(3):
    compiled(x)
print("Warmup done.")

compiled_timer = Timer(
    stmt="fn(x)",
    globals={"fn": compiled, "x": x},
    label="FFN",
    sub_label=f"dim={dim}",
    description="compiled",
    num_threads=1,
)

compile_results = [
    eager_timer.blocked_autorange(min_run_time=1.0),
    compiled_timer.blocked_autorange(min_run_time=1.0),
]

compare = Compare(compile_results)
compare.colorize()
compare.print()

speedup = compile_results[0].median / compile_results[1].median
print(f"\ntorch.compile speedup: {speedup:.2f}x")

## 8. `num_threads` Control

CPU benchmarks vary with thread count. Pin it for reproducibility.

In [ ]:
print(f"Default threads: {torch.get_num_threads()}")

x_big = torch.randn(1000, 1000)
y_big = torch.randn(1000, 1000)

thread_results = []
max_t = min(torch.get_num_threads(), 8)
thread_counts = [t for t in [1, 2, 4, 8] if t <= max_t]

for nt in thread_counts:
    t = Timer(
        stmt="x @ y",
        globals={"x": x_big, "y": y_big},
        num_threads=nt,
        label="Threading",
        sub_label="[1000x1000]",
        description=f"{nt} thread{'s' if nt > 1 else ''}",
    )
    thread_results.append(t.blocked_autorange(min_run_time=0.5))

Compare(thread_results).print()

base = thread_results[0].median
for i, nt in enumerate(thread_counts):
    print(f"  {nt} thread(s): {base / thread_results[i].median:.2f}x speedup")

## 9. Common Pitfalls (Reference)

| Pitfall | Problem | Solution |
|---------|---------|----------|
| No warmup | Cold start overhead | `blocked_autorange()` handles it |
| No CUDA sync | Measures launch time only | `Timer` syncs automatically |
| GC interference | Random latency spikes | `Timer` disables GC |
| Too few iterations | Noisy results | `min_run_time=2.0` |
| In-place vs out-of-place | Unfair allocation comparison | Be explicit about what you're comparing |
| Unpinned threads | Non-reproducible CPU results | Use `num_threads` parameter |
| Including compile time | Measures compilation, not runtime | Warmup compiled models first |

In [ ]:
# Pitfall demonstration: in-place vs out-of-place
x_pit = torch.randn(1000, 1000)

oop = Timer(
    stmt="x + 1",
    globals={"x": x_pit},
    label="In-place pitfall",
    sub_label="[1000x1000]",
    description="out-of-place",
    num_threads=1,
)
ip = Timer(
    stmt="x.add_(1)",
    globals={"x": x_pit.clone()},
    label="In-place pitfall",
    sub_label="[1000x1000]",
    description="in-place",
    num_threads=1,
)

pit_results = [oop.blocked_autorange(min_run_time=0.5), ip.blocked_autorange(min_run_time=0.5)]
Compare(pit_results).print()
print("In-place is faster partly due to zero allocation overhead.")

## 10. Fuzzer — Random Test Configurations

Avoid cherry-picking benchmark sizes. The `Fuzzer` generates random shapes, contiguity, and parameters.

In [ ]:
from torch.utils.benchmark import Fuzzer, FuzzedParameter, FuzzedTensor

fuzzer = Fuzzer(
    parameters=[
        FuzzedParameter("k", minval=4, maxval=10, distribution="uniform"),
        FuzzedParameter("m", minval=4, maxval=10, distribution="uniform"),
    ],
    tensors=[
        FuzzedTensor("x", size=("k", "m"), probability_contiguous=0.6),
        FuzzedTensor("y", size=("m", "k"), probability_contiguous=0.6),
    ],
    seed=42,
)

fuzz_results = []
for tensors, tensor_params, params in fuzzer.take(6):
    k, m = int(params["k"]), int(params["m"])
    xc = tensors["x"].is_contiguous()
    yc = tensors["y"].is_contiguous()

    t = Timer(
        stmt="x @ y",
        globals=tensors,
        label="Fuzzed matmul",
        sub_label=f"[{k}x{m}]@[{m}x{k}] contig=({xc},{yc})",
        description="mm",
        num_threads=1,
    )
    fuzz_results.append(t.blocked_autorange(min_run_time=0.2))

Compare(fuzz_results).print()

## 11. Dtype Comparison

Compare performance across data types.

In [ ]:
n = 1024
dtype_results = []

for name, dtype in [("float32", torch.float32), ("float16", torch.float16), ("bfloat16", torch.bfloat16)]:
    a = torch.randn(n, n, dtype=dtype)
    b = torch.randn(n, n, dtype=dtype)
    t = Timer(
        stmt="a @ b",
        globals={"a": a, "b": b},
        label="Dtype matmul",
        sub_label=f"[{n}x{n}]",
        description=name,
        num_threads=1,
    )
    dtype_results.append(t.blocked_autorange(min_run_time=0.5))

compare = Compare(dtype_results)
compare.colorize()
compare.print()

fp32_time = dtype_results[0].median
for i, (name, _) in enumerate([("float32", None), ("float16", None), ("bfloat16", None)]):
    print(f"  {name}: {fp32_time / dtype_results[i].median:.2f}x vs float32")

## 12. Full Recipe: Model Comparison

Compare two model implementations — forward and forward+backward.

In [ ]:
class FFN_ReLU(nn.Module):
    def __init__(self, dim, depth=3):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers.extend([nn.Linear(dim, dim), nn.ReLU()])
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class FFN_GELU(nn.Module):
    def __init__(self, dim, depth=3):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers.extend([nn.Linear(dim, dim), nn.GELU()])
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

mdim = 256
criterion = nn.MSELoss()
models_map = {"ReLU": FFN_ReLU(mdim), "GELU": FFN_GELU(mdim)}

model_results = []
for batch in [16, 64, 256]:
    inp = torch.randn(batch, mdim)
    tgt = torch.randn(batch, mdim)
    for name, mdl in models_map.items():
        t = Timer(
            stmt="model(x)",
            globals={"model": mdl, "x": inp},
            label="Activation compare",
            sub_label=f"batch={batch}",
            description=f"{name} fwd",
            num_threads=1,
        )
        model_results.append(t.blocked_autorange(min_run_time=0.5))

compare = Compare(model_results)
compare.trim_significant_figures()
compare.colorize()
compare.print()

## 13. Multi-Statement Benchmarks

Benchmark forward + backward together.

In [ ]:
model_fb = FFN_GELU(256)
criterion_fb = nn.MSELoss()
x_fb = torch.randn(64, 256)
t_fb = torch.randn(64, 256)

fwd_only = Timer(
    stmt="model(x)",
    globals={"model": model_fb, "x": x_fb},
    label="Phases",
    sub_label="GELU-FFN",
    description="forward",
    num_threads=1,
)

fwd_bwd = Timer(
    stmt="y = model(x)\nloss = crit(y, tgt)\nloss.backward()",
    globals={"model": model_fb, "crit": criterion_fb, "x": x_fb, "tgt": t_fb},
    label="Phases",
    sub_label="GELU-FFN",
    description="fwd+bwd",
    num_threads=1,
)

phase_results = [
    fwd_only.blocked_autorange(min_run_time=0.5),
    fwd_bwd.blocked_autorange(min_run_time=0.5),
]
Compare(phase_results).print()

ratio = phase_results[1].median / phase_results[0].median
print(f"\nfwd+bwd / fwd ratio: {ratio:.2f}x")

## 14. Custom Function Benchmarking

Compare your own implementations against PyTorch built-ins.

In [ ]:
def softmax_naive(x):
    e = torch.exp(x - x.max(dim=-1, keepdim=True).values)
    return e / e.sum(dim=-1, keepdim=True)

def softmax_logsumexp(x):
    return torch.exp(x - torch.logsumexp(x, dim=-1, keepdim=True))

x_sm = torch.randn(256, 1024)

sm_results = []
for desc, fn, stmt in [
    ("torch.softmax", torch.nn.functional.softmax, "fn(x, dim=-1)"),
    ("naive", softmax_naive, "fn(x)"),
    ("logsumexp", softmax_logsumexp, "fn(x)"),
]:
    t = Timer(
        stmt=stmt,
        globals={"fn": fn, "x": x_sm},
        label="Softmax",
        sub_label="[256x1024]",
        description=desc,
        num_threads=1,
    )
    sm_results.append(t.blocked_autorange(min_run_time=0.5))

compare = Compare(sm_results)
compare.colorize()
compare.print()

---

## Exercise

**Benchmark matmul for sizes 128 to 4096, and analyze the scaling.**

1. Create Timer objects for sizes `[128, 256, 512, 1024, 2048, 4096]`
2. Use `blocked_autorange(min_run_time=0.5)`
3. Display a Compare table
4. Print the scaling ratios (actual vs theoretical O(n^3))

In [ ]:
# Exercise: fill in the code
exercise_sizes = [128, 256, 512, 1024, 2048, 4096]
exercise_results = []

for n in exercise_sizes:
    # TODO: create tensors, Timer, and collect results
    pass

# TODO: Compare(exercise_results).print()
# TODO: print scaling ratios

In [ ]:
# Solution
exercise_sizes = [128, 256, 512, 1024, 2048, 4096]
exercise_results = []

for n in exercise_sizes:
    a = torch.randn(n, n)
    b = torch.randn(n, n)
    t = Timer(
        stmt="a @ b",
        globals={"a": a, "b": b},
        label="Exercise: matmul",
        sub_label=f"[{n}x{n}]",
        description="mm",
        num_threads=1,
    )
    exercise_results.append(t.blocked_autorange(min_run_time=0.5))

Compare(exercise_results).print()

print("\nScaling analysis:")
for i in range(1, len(exercise_sizes)):
    actual = exercise_results[i].median / exercise_results[i - 1].median
    theory = (exercise_sizes[i] / exercise_sizes[i - 1]) ** 3
    print(f"  {exercise_sizes[i]:4d} vs {exercise_sizes[i-1]:4d}: "
          f"actual={actual:.1f}x, O(n^3)={theory:.1f}x")

## Key Takeaways

1. **Never use `time.time()`** for PyTorch benchmarks — use `torch.utils.benchmark.Timer`
2. **`blocked_autorange()`** is the recommended method — handles warmup, GC, iteration count
3. **Median over mean** — resistant to outliers from GC, scheduling, thermals
4. **`Compare` tables** organize multi-config benchmarks with `label`/`sub_label`/`description`
5. **Warmup `torch.compile`** before measuring — compilation time is not runtime
6. **Pin `num_threads`** for reproducible CPU benchmarks
7. **`Fuzzer`** avoids cherry-picking benchmark inputs
8. **Callgrind** gives deterministic instruction counts for micro-benchmarks (requires Valgrind)

---

**Next Steps:**
- [Module 07 — Training Pipelines](../07_training/) for models to benchmark
- [Module 08 — torch.compile](../08_torch_compile/) for understanding compilation
- [Module 26 — Memory Profiling](../26_memory_profiling/) for complementary profiling tools